## **Redes Bayesianas**

Uma **Rede Bayesiana** é um modelo probabilístico definido por um par $(\mathcal{G}, \Theta)$, onde:

* **$\mathcal{G}$**: um **Grafo Acíclico Direcionado (DAG)**
* **$\Theta$**: conjunto de parâmetros das distribuições condicionais

Cada nó $X_i$ representa uma variável aleatória, e cada aresta representa uma dependência condicional direta.

Seja $X = (X_1, X_2, \dots, X_n)$. A distribuição conjunta fatoriza é dada por:

$$
P(X_1, X_2, \dots, X_n) = \prod_{i=1}^{n} P(X_i \mid \mathrm{Pa}(X_i))
$$

onde $\mathrm{Pa}(X_i)$ é o conjunto de pais de $X_i$ no grafo.

Esse modelo deseja superar Naive Bayes ao passo que neste último, assumimos — de forma ingênua — a independência condicional total:

$$
P(X_1, X_2, \dots, X_n \mid Y) = \prod_{i=1}^{n} P(X_i \mid Y)
$$

ou seja, $X_i \perp X_j \mid Y$ para todo $i \neq j$.

**Redes Bayesianas** generalizam essa hipótese permitindo dependências adicionais entre variáveis:

$$
P(Y, X_1, \dots, X_n) := P(Y) \prod_{i=1}^{n} P(X_i \mid \mathrm{Pa}(X_i))
$$

### **Fundamentação Probabilística**

Seja $X = (X_1, \dots, X_n)$ e uma variável de classe $Y$.

Pela regra da cadeia das  probabilidades:

$$
P(Y, X_1, \dots, X_n) = P(Y) \prod_{i=1}^{n} P(X_i \mid Y, X_1, \dots, X_{i-1})
$$

Em Redes Bayesianas, essa fatoração é simplificada usando o grafo:

$$
P(Y, X_1, \dots, X_n) := P(Y) \prod_{i=1}^{n} P(X_i \mid \mathrm{Pa}(X_i))
$$

onde $\mathrm{Pa}(X_i)$ depende da estrutura $\mathcal{G}$.

### **Aprendizado em Redes Bayesianas**

O aprendizado envolve duas etapas principais:

#### **Aprendizado da Estrutura**

Determinar o DAG que melhor representa as dependências entre as variáveis:

$$
\mathcal{G}^* = \arg\max_{\mathcal{G}} \ \text{Score}(\mathcal{G} \mid \mathcal{D})
$$

onde $\mathcal{D}$ é o conjunto de dados.

Funções de score comuns:

* Log-verossimilhança
* BIC (Bayesian Information Criterion)
* MDL (Minimum Description Length)

#### **Aprendizado dos Parâmetros**

Dada uma estrutura $\mathcal{G}$, o objetivo é estimar as distribuições de probabilidade condicionais associadas a cada nó:

$$
P(X_i \mid \mathrm{Pa}(X_i))
$$

Esses parâmetros podem ser estimados por:

* **Máxima Verossimilhança (MLE)**:

$$
\hat{P}(x_i \mid \pi_i) = \frac{N(x_i, \pi_i)}{N(\pi_i)}
$$

onde:
* $N(x_i, \pi_i)$ é a contagem conjunta
* $N(\pi_i)$ é a contagem dos pais

* **Suavização de Laplace**:

$$
\hat{P}(x_i \mid \pi_i) = \frac{N(x_i, \pi_i) + \alpha}{N(\pi_i) + \alpha k}
$$

onde:
* $\alpha > 0$ é o hiperparâmetro
* $k$ é o número de valores possíveis de $X_i$

Note que essa abordagem é para variáveis **discretas**.

### **Classificadores Bayesianos com $k$ dependências ($k$-DB)**

Cada variável $X_i$ pode depender:

* Da classe $Y$
* De no máximo $k$ outras variáveis

Logo:

$$
P(X_i \mid Y, \pi_i)
$$

com:

$$
\pi_i \subseteq \{X_1, \dots, X_n\}, \quad |\pi_i| \leq k
$$

onde $\pi_i = \mathrm{Pa}(X_i)$ representa o conjunto de pais de $X_i$ no grafo.

### **Implementação de Redes Bayesianas com $k$ dependências para problema de Classificação**

Vamos utilizar **Redes Bayesianas com $k$ dependências** para tarefa de **classificação**  no dataset [**Car Evaluation**](https://archive.ics.uci.edu/dataset/19/car+evaluation).

In [23]:
!pip install ucimlrepo

In [24]:
from ucimlrepo import fetch_ucirepo

car_evaluation = fetch_ucirepo(id=19)

X = car_evaluation.data.features
y = car_evaluation.data.targets

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

Shape de X: (1728, 6)
Shape de y: (1728, 1)


In [25]:
df = X.copy()
df['class'] = y

df.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [26]:
df["class"].value_counts()

,count
class,
unacc,1210
acc,384
good,69
vgood,65


In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (1382, 6)
Teste: (346, 6)
